# 01_check_pdf_extract

## 목적

이 노트북은 구조화 RAG 구축 전, `data/vectordb/`에 있는 규정 PDF들이 정상적으로 텍스트 추출되는지 확인한다.

확인 항목:

1. PDF 파일 목록 확인
2. PDF 페이지 수 확인
3. 페이지별 텍스트 추출 가능 여부 확인
4. 조문 패턴 `제○조(제목)` 탐지 가능 여부 확인
5. 텍스트가 거의 없는 스캔 PDF 여부 확인
6. 이후 parser 개발에 사용할 샘플 텍스트 저장

이 노트북에서는 아직 Vector DB를 만들지 않는다.
이 노트북에서는 아직 BM25도 만들지 않는다.
이 노트북의 결과는 Phase 1 regulation parser 개발의 입력 진단용으로만 사용한다.

In [1]:
# 기본 라이브러리 및 경로 설정
from pathlib import Path
import re
import json
from pprint import pprint
from typing import List, Dict, Any

try:
    import fitz  # PyMuPDF
except ImportError:
    raise ImportError(
        "PyMuPDF가 설치되어 있지 않습니다. 아래 명령을 실행하세요:\n"
        "%pip install pymupdf"
    )


# 현재 노트북이 notebooks/ 안에서 실행된다고 가정
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PDF_DIR = PROJECT_ROOT / "data" / "vectordb"
RETRIEVAL_DIR = PROJECT_ROOT / "data" / "retrieval"
DEBUG_DIR = RETRIEVAL_DIR / "debug_pdf_extract"

COLLECTION_NAME = "complypilot_regulations_v2"

DEBUG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PDF_DIR:", PDF_DIR)
print("RETRIEVAL_DIR:", RETRIEVAL_DIR)
print("DEBUG_DIR:", DEBUG_DIR)
print("COLLECTION_NAME:", COLLECTION_NAME)

PROJECT_ROOT: c:\Users\USER\Desktop\complypilot-jb
PDF_DIR: c:\Users\USER\Desktop\complypilot-jb\data\vectordb
RETRIEVAL_DIR: c:\Users\USER\Desktop\complypilot-jb\data\retrieval
DEBUG_DIR: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_pdf_extract
COLLECTION_NAME: complypilot_regulations_v2


In [2]:
# PDF 파일 목록 확인
pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print("PDF 개수:", len(pdf_files))

for idx, pdf_path in enumerate(pdf_files, start=1):
    print(f"{idx:02d}. {pdf_path.name}")

PDF 개수: 8
01. 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf
02. 금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428).pdf
03. 금융소비자 보호에 관한 법률(법률)(제21065호)(20260102).pdf
04. 여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506).pdf
05. 여신전문금융업법(법률)(제21065호)(20251001).pdf
06. 예금자보호법(법률)(제21065호)(20260102).pdf
07. 은행업감독규정 (금융위원회고시)(제2026-10호)(20260401).pdf
08. 표시ㆍ광고의 공정화에 관한 법률(법률)(제20712호)(20250121).pdf


In [3]:
# PDF 기본 정보 확인
def get_pdf_basic_info(pdf_path: Path) -> Dict[str, Any]:
    """
    PDF 파일의 기본 정보를 확인합니다.

    Args:
        pdf_path: PDF 파일 경로

    Return:
        PDF 파일명, 페이지 수, 파일 크기를 담은 dict
    """
    doc = fitz.open(pdf_path)
    
    info = {
        "file_name": pdf_path.name,
        "page_count": len(doc),
        "file_size_mb": round(pdf_path.stat().st_size / (1024 * 1024), 2),
    }
    
    doc.close()
    return info


pdf_infos = []

for pdf_path in pdf_files:
    try:
        pdf_infos.append(get_pdf_basic_info(pdf_path))
    except Exception as e:
        pdf_infos.append({
            "file_name": pdf_path.name,
            "error": str(e)
        })

pprint(pdf_infos[:10])

[{'file_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
  'file_size_mb': 0.17,
  'page_count': 28},
 {'file_name': '금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428).pdf',
  'file_size_mb': 0.19,
  'page_count': 28},
 {'file_name': '금융소비자 보호에 관한 법률(법률)(제21065호)(20260102).pdf',
  'file_size_mb': 0.19,
  'page_count': 25},
 {'file_name': '여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506).pdf',
  'file_size_mb': 0.19,
  'page_count': 34},
 {'file_name': '여신전문금융업법(법률)(제21065호)(20251001).pdf',
  'file_size_mb': 0.26,
  'page_count': 31},
 {'file_name': '예금자보호법(법률)(제21065호)(20260102).pdf',
  'file_size_mb': 0.19,
  'page_count': 22},
 {'file_name': '은행업감독규정 (금융위원회고시)(제2026-10호)(20260401).pdf',
  'file_size_mb': 0.26,
  'page_count': 57},
 {'file_name': '표시ㆍ광고의 공정화에 관한 법률(법률)(제20712호)(20250121).pdf',
  'file_size_mb': 0.12,
  'page_count': 7}]


In [4]:
# 샘플 PDF 선택
if not pdf_files:
    raise FileNotFoundError(f"PDF 파일이 없습니다: {PDF_DIR}")

sample_pdf = pdf_files[0]

print("선택된 샘플 PDF:", sample_pdf.name)

선택된 샘플 PDF: 금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf


In [5]:
# 페이지별 텍스트 추출 미리보기
def extract_page_texts(pdf_path: Path, max_pages: int | None = None) -> List[Dict[str, Any]]:
    """
    PDF에서 페이지별 텍스트를 추출합니다.

    Args:
        pdf_path: PDF 파일 경로
        max_pages: 추출할 최대 페이지 수. None이면 전체 페이지 추출

    Return:
        page, text, text_length를 포함한 리스트
    """
    doc = fitz.open(pdf_path)
    page_texts = []
    
    total_pages = len(doc)
    limit = total_pages if max_pages is None else min(max_pages, total_pages)
    
    for page_idx in range(limit):
        page = doc[page_idx]
        text = page.get_text("text")
        
        page_texts.append({
            "page": page_idx + 1,
            "text": text,
            "text_length": len(text.strip()),
        })
    
    doc.close()
    return page_texts


sample_page_texts = extract_page_texts(sample_pdf, max_pages=5)

for item in sample_page_texts:
    print("=" * 100)
    print(f"PAGE: {item['page']} | text_length: {item['text_length']}")
    print("-" * 100)
    print(item["text"][:2000])

PAGE: 1 | text_length: 1434
----------------------------------------------------------------------------------------------------
법제처                                                            1                                                   국가법령정보센터
금융소비자 보호에 관한 감독규정
금융소비자 보호에 관한 감독규정
[시행 2026. 4. 2.] [금융위원회고시 제2026-11호, 2026. 4. 2., 일부개정]
 
금융위원회(금융소비자정책과), 02-2100-2524
 
제1조(목적) 이 규정은 「금융소비자 보호에 관한 법률」 및 같은 법 시행령에서 위임하는 사항과 그 시행에 필요한
사항을 규정함을 목적으로 한다.
 
제2조(정의) ① 「금융소비자 보호에 관한 법률 시행령」(이하 "영"이라 한다) 제2조제1항제7호에서 "금융위원회가
정하여 고시하는 것"이란 다음 각 호의 어느 하나에 해당하는 것을 말한다.
1. 다음 각 목의 자가 계약에 따라 금융소비자로부터 금전을 받고 장래에 그 금전과 그에 따른 이자 등의 대가를
지급하기로 하는 계약. 다만, 「주택법」에 따른 입주자저축은 제외한다.
가. 「금융산업의 구조개선에 관한 법률」에 따라 「자본시장과 금융투자업에 관한 법률」에 따른 종합금융회사
와 합병한 기관(「예금자보호법」 제2조제1호가목부터 사목까지의 부보금융회사를 말한다)
나. 「농업협동조합법」에 따른 농협은행
다. 「상호저축은행법」에 따른 상호저축은행
라. 「수산업협동조합법」에 따른 수협은행
마. 「신용협동조합법」에 따른 조합(이하 "신용협동조합"이라 한다)
바. 「은행법」에 따라 인가를 받은 은행
사. 「자본시장과 금융투자업에 관한 법률」에 따른 금융투자업자 및 증권금융회사
아. 「자본시장과 금융투자업에 관한 법률」에 따른 종합금융회사
자. 「중소기업은행법

In [6]:
# 전체 PDF 텍스트 추출 가능성 진단
def diagnose_pdf_text_quality(pdf_path: Path) -> Dict[str, Any]:
    """
    PDF의 텍스트 추출 품질을 간단히 진단합니다.

    Args:
        pdf_path: PDF 파일 경로

    Return:
        페이지 수, 총 텍스트 길이, 평균 텍스트 길이, 빈 페이지 수 등을 담은 dict
    """
    doc = fitz.open(pdf_path)
    
    page_lengths = []
    
    for page_idx in range(len(doc)):
        text = doc[page_idx].get_text("text").strip()
        page_lengths.append(len(text))
    
    doc.close()
    
    total_pages = len(page_lengths)
    total_text_length = sum(page_lengths)
    empty_pages = sum(1 for length in page_lengths if length < 30)
    avg_text_length = round(total_text_length / total_pages, 2) if total_pages else 0
    
    if total_text_length == 0:
        quality = "텍스트 추출 불가 가능성 높음"
    elif empty_pages / total_pages > 0.5:
        quality = "스캔 PDF 또는 추출 품질 낮음 가능성"
    elif avg_text_length < 300:
        quality = "텍스트 적음 / 표지 또는 목차 중심 가능성"
    else:
        quality = "텍스트 추출 가능"
    
    return {
        "file_name": pdf_path.name,
        "page_count": total_pages,
        "total_text_length": total_text_length,
        "avg_text_length": avg_text_length,
        "empty_or_short_pages": empty_pages,
        "quality": quality,
    }


diagnosis = [diagnose_pdf_text_quality(pdf_path) for pdf_path in pdf_files]

for row in diagnosis:
    print(row)

{'file_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf', 'page_count': 28, 'total_text_length': 42194, 'avg_text_length': 1506.93, 'empty_or_short_pages': 0, 'quality': '텍스트 추출 가능'}
{'file_name': '금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428).pdf', 'page_count': 28, 'total_text_length': 50770, 'avg_text_length': 1813.21, 'empty_or_short_pages': 0, 'quality': '텍스트 추출 가능'}
{'file_name': '금융소비자 보호에 관한 법률(법률)(제21065호)(20260102).pdf', 'page_count': 25, 'total_text_length': 48239, 'avg_text_length': 1929.56, 'empty_or_short_pages': 0, 'quality': '텍스트 추출 가능'}
{'file_name': '여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506).pdf', 'page_count': 34, 'total_text_length': 53655, 'avg_text_length': 1578.09, 'empty_or_short_pages': 0, 'quality': '텍스트 추출 가능'}
{'file_name': '여신전문금융업법(법률)(제21065호)(20251001).pdf', 'page_count': 31, 'total_text_length': 57278, 'avg_text_length': 1847.68, 'empty_or_short_pages': 0, 'quality': '텍스트 추출 가능'}
{'file_name': '예금자보호법(법률)(제21065호)(20260102).pdf', 'page_count':

In [7]:
# 조문 패턴 탐지 테스트
ARTICLE_PATTERN = re.compile(
    r"제\s*\d+\s*조(?:의\s*\d+)?\s*\([^)]*\)"
)

def find_article_headers_in_text(text: str) -> List[str]:
    """
    텍스트에서 조문 헤더 패턴을 찾습니다.

    Args:
        text: PDF에서 추출한 텍스트

    Return:
        탐지된 조문 헤더 리스트
    """
    return ARTICLE_PATTERN.findall(text)


for item in sample_page_texts:
    headers = find_article_headers_in_text(item["text"])
    
    if headers:
        print("=" * 100)
        print("PAGE:", item["page"])
        print(headers[:20])

PAGE: 1
['제1조(목적)', '제2조(정의)']
PAGE: 3
['제3조(금융상품의 유형)']
PAGE: 4
['제4조(금융회사등의 업종구분)', '제5조(금융상품자문업자의 등록요건)']
PAGE: 5
['제6조(금융상품판매대리ㆍ중개업자의 등록요건)']


In [8]:
# 전체 PDF에서 조문 후보 수집
def collect_article_candidates(pdf_path: Path) -> List[Dict[str, Any]]:
    """
    PDF 전체에서 조문 헤더 후보를 수집합니다.

    Args:
        pdf_path: PDF 파일 경로

    Return:
        file_name, page, article_header를 담은 리스트
    """
    doc = fitz.open(pdf_path)
    candidates = []
    
    for page_idx in range(len(doc)):
        text = doc[page_idx].get_text("text")
        headers = find_article_headers_in_text(text)
        
        for header in headers:
            candidates.append({
                "file_name": pdf_path.name,
                "page": page_idx + 1,
                "article_header": header,
            })
    
    doc.close()
    return candidates


all_article_candidates = []

for_pdf_count = 0
for pdf_path in pdf_files:
    try:
        candidates = collect_article_candidates(pdf_path)
        all_article_candidates.extend(candidates)
        for_pdf_count += 1
    except Exception as e:
        print(f"[ERROR] {pdf_path.name}: {e}")

print("처리 PDF 수:", for_pdf_count)
print("전체 조문 후보 수:", len(all_article_candidates))

pprint(all_article_candidates[:30])

처리 PDF 수: 8
전체 조문 후보 수: 608
[{'article_header': '제1조(목적)',
  'file_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
  'page': 1},
 {'article_header': '제2조(정의)',
  'file_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
  'page': 1},
 {'article_header': '제3조(금융상품의 유형)',
  'file_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
  'page': 3},
 {'article_header': '제4조(금융회사등의 업종구분)',
  'file_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
  'page': 4},
 {'article_header': '제5조(금융상품자문업자의 등록요건)',
  'file_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
  'page': 4},
 {'article_header': '제6조(금융상품판매대리ㆍ중개업자의 등록요건)',
  'file_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
  'page': 5},
 {'article_header': '제7조(등록신청)',
  'file_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
  'page': 6},
 {'article_header': '제8조(등록수수료)',
  'file_name': '금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf',
  'page': 6},
 {'

In [9]:
# PDF별 조문 후보 개수 요약
from collections import Counter

article_count_by_file = Counter(
    item["file_name"] for item in all_article_candidates
)

for file_name, count in article_count_by_file.most_common():
    print(f"{file_name}: {count}개")

은행업감독규정 (금융위원회고시)(제2026-10호)(20260401).pdf: 132개
여신전문금융업법(법률)(제21065호)(20251001).pdf: 112개
여신전문금융업감독규정(금융위원회고시)(제2026-17호)(20260506).pdf: 88개
예금자보호법(법률)(제21065호)(20260102).pdf: 86개
금융소비자 보호에 관한 법률(법률)(제21065호)(20260102).pdf: 75개
금융소비자 보호에 관한 법률 시행령(대통령령)(제36287호)(20260428).pdf: 53개
금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402).pdf: 38개
표시ㆍ광고의 공정화에 관한 법률(법률)(제20712호)(20250121).pdf: 24개


In [10]:
# 조문 헤더 파싱 테스트
ARTICLE_PARSE_PATTERN = re.compile(
    r"(제\s*\d+\s*조(?:의\s*\d+)?)\s*\(([^)]*)\)"
)

def parse_article_header(header: str) -> Dict[str, str]:
    """
    조문 헤더에서 조문번호와 조문제목을 분리합니다.

    Args:
        header: 예) 제17조(설명의무)

    Return:
        article_no, article_title을 담은 dict
    """
    match = ARTICLE_PARSE_PATTERN.search(header)
    
    if not match:
        return {
            "article_no": "",
            "article_title": "",
            "raw_header": header,
        }
    
    article_no = re.sub(r"\s+", "", match.group(1))
    article_title = match.group(2).strip()
    
    return {
        "article_no": article_no,
        "article_title": article_title,
        "raw_header": header,
    }


parsed_headers = [
    parse_article_header(item["article_header"])
    for item in all_article_candidates[:30]
]

pprint(parsed_headers)

[{'article_no': '제1조', 'article_title': '목적', 'raw_header': '제1조(목적)'},
 {'article_no': '제2조', 'article_title': '정의', 'raw_header': '제2조(정의)'},
 {'article_no': '제3조',
  'article_title': '금융상품의 유형',
  'raw_header': '제3조(금융상품의 유형)'},
 {'article_no': '제4조',
  'article_title': '금융회사등의 업종구분',
  'raw_header': '제4조(금융회사등의 업종구분)'},
 {'article_no': '제5조',
  'article_title': '금융상품자문업자의 등록요건',
  'raw_header': '제5조(금융상품자문업자의 등록요건)'},
 {'article_no': '제6조',
  'article_title': '금융상품판매대리ㆍ중개업자의 등록요건',
  'raw_header': '제6조(금융상품판매대리ㆍ중개업자의 등록요건)'},
 {'article_no': '제7조', 'article_title': '등록신청', 'raw_header': '제7조(등록신청)'},
 {'article_no': '제8조', 'article_title': '등록수수료', 'raw_header': '제8조(등록수수료)'},
 {'article_no': '제9조', 'article_title': '내부통제기준', 'raw_header': '제9조(내부통제기준)'},
 {'article_no': '제10조',
  'article_title': '적합성 원칙',
  'raw_header': '제10조(적합성 원칙)'},
 {'article_no': '제11조', 'article_title': '적정성원칙', 'raw_header': '제11조(적정성원칙)'},
 {'article_no': '제12조', 'article_title': '설명의무', 'raw_header': '

In [11]:
# 샘플 PDF 텍스트를 디버그 파일로 저장
def save_pdf_text_debug_file(pdf_path: Path, output_dir: Path) -> Path:
    """
    PDF 전체 텍스트를 디버그용 txt 파일로 저장합니다.

    Args:
        pdf_path: PDF 파일 경로
        output_dir: 저장 디렉터리

    Return:
        저장된 txt 파일 경로
    """
    page_texts = extract_page_texts(pdf_path, max_pages=None)
    
    output_path = output_dir / f"{pdf_path.stem}__debug_text.txt"
    
    with open(output_path, "w", encoding="utf-8") as f:
        for item in page_texts:
            f.write("=" * 100 + "\n")
            f.write(f"PAGE: {item['page']} | text_length: {item['text_length']}\n")
            f.write("=" * 100 + "\n")
            f.write(item["text"])
            f.write("\n\n")
    
    return output_path


debug_text_path = save_pdf_text_debug_file(sample_pdf, DEBUG_DIR)

print("저장 완료:", debug_text_path)

저장 완료: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_pdf_extract\금융소비자 보호에 관한 감독규정(금융위원회고시)(제2026-11호)(20260402)__debug_text.txt


In [12]:
# 전체 PDF 진단 결과 저장
diagnosis_path = DEBUG_DIR / "pdf_text_quality_diagnosis.json"
article_candidates_path = DEBUG_DIR / "article_candidates.json"

with open(diagnosis_path, "w", encoding="utf-8") as f:
    json.dump(diagnosis, f, ensure_ascii=False, indent=2)

with open(article_candidates_path, "w", encoding="utf-8") as f:
    json.dump(all_article_candidates, f, ensure_ascii=False, indent=2)

print("PDF 텍스트 품질 진단 저장:", diagnosis_path)
print("조문 후보 저장:", article_candidates_path)

PDF 텍스트 품질 진단 저장: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_pdf_extract\pdf_text_quality_diagnosis.json
조문 후보 저장: c:\Users\USER\Desktop\complypilot-jb\data\retrieval\debug_pdf_extract\article_candidates.json


In [13]:
# 최종 체크 요약
summary = {
    "pdf_count": len(pdf_files),
    "diagnosed_pdf_count": len(diagnosis),
    "article_candidate_count": len(all_article_candidates),
    "debug_dir": str(DEBUG_DIR),
    "collection_name": COLLECTION_NAME,
}

pprint(summary)

if len(pdf_files) == 0:
    print("[FAIL] data/vectordb/에 PDF가 없습니다.")
elif len(all_article_candidates) == 0:
    print("[WARN] 조문 후보가 탐지되지 않았습니다. PDF 텍스트 구조 또는 정규식 보완이 필요합니다.")
else:
    print("[OK] PDF 텍스트 추출 및 조문 후보 탐지 1차 확인 완료")

{'article_candidate_count': 608,
 'collection_name': 'complypilot_regulations_v2',
 'debug_dir': 'c:\\Users\\USER\\Desktop\\complypilot-jb\\data\\retrieval\\debug_pdf_extract',
 'diagnosed_pdf_count': 8,
 'pdf_count': 8}
[OK] PDF 텍스트 추출 및 조문 후보 탐지 1차 확인 완료
